In [1]:
# spending_clustering.ipynb
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Optional: for consistent output
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [4]:
# Load CSV
df = pd.read_csv("dataset/spending_l9_dataset.csv")

# Preview
df.head()

,CustomerID,Age,Income_$,SpendingScore,VisitsPerMonth,OnlinePurchases,Gender,Region
0,1,28,33,78,14,9,Female,East
1,2,21,25,87,8,23,Male,North
2,3,23,24,88,13,10,Male,South
3,4,24,25,73,16,11,Female,West
4,5,20,23,88,17,16,Male,West


In [6]:
# Features
X = df[["Income_$", "SpendingScore"]].copy()

# Handle missing values (median)
X.fillna(X.median(), inplace=True)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [7]:
print("=== SSE for k=1..10 ===")
sse = []
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    sse.append(kmeans.inertia_)
    print(f"k={k} → SSE={kmeans.inertia_:.2f}")

=== SSE for k=1..10 ===
k=1 → SSE=400.00
k=2 → SSE=199.70
k=3 → SSE=77.01
k=4 → SSE=21.37
k=5 → SSE=17.93
k=6 → SSE=15.65
k=7 → SSE=13.88
k=8 → SSE=12.45
k=9 → SSE=11.06
k=10 → SSE=9.93


In [8]:
# Pick K based on SSE trend (example: K=4)
K = 4
kmeans_final = KMeans(n_clusters=K, random_state=42, n_init=10)
labels = kmeans_final.fit_predict(X_scaled)

# Add cluster labels to DataFrame
df['Cluster'] = labels

In [9]:
# Silhouette and Davies–Bouldin
sil_score = silhouette_score(X_scaled, labels)
dbi_score = davies_bouldin_score(X_scaled, labels)

print(f"Silhouette Score : {sil_score:.3f}")
print(f"Davies–Bouldin   : {dbi_score:.3f}")

Silhouette Score : 0.729
Davies–Bouldin   : 0.387


In [10]:
# Inverse-transform centers to original scale
centers_orig = scaler.inverse_transform(kmeans_final.cluster_centers_)
centers_df = pd.DataFrame(centers_orig, columns=["Income_$", "SpendingScore"])
centers_df.index.name = 'Cluster'
centers_df = centers_df.round(2)

print("\n=== CLUSTER CENTERS (Original Units) ===")
print(centers_df)


=== CLUSTER CENTERS (Original Units) ===
         Income_$  SpendingScore
Cluster                         
0           56.32          53.58
1           28.92          19.60
2           24.14          83.10
3           99.16          79.24


In [11]:
print("\n=== Sample Rows with Cluster Labels ===")
print(df[["Income_$", "SpendingScore", "Cluster"]].sample(3))


=== Sample Rows with Cluster Labels ===
     Income_$  SpendingScore  Cluster
53         60             38        0
175        97             78        3
96         61             55        0


In [12]:
df.to_csv("spending_labeled_clusters.csv", index=False)